In [1]:
from pathlib import Path

import joblib
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

In [2]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.net(x)

In [3]:
rf = joblib.load("models/iris_rf.pkl")
initial_type = [("float_input", FloatTensorType([None, 4]))]
onx = convert_sklearn(rf, initial_types=initial_type)
with open("models/iris_rf.onnx", "wb") as f:
    f.write(onx.SerializeToString())
print("iris_rf.onnx saved")

iris_rf.onnx saved


In [4]:
cat = CatBoostClassifier()
cat.load_model("models/iris_catboost.cbm")
cat.save_model("models/iris_catboost.onnx", format="onnx")
print("iris_catboost.onnx saved")

iris_catboost.onnx saved


In [5]:
model = IrisNet()
model.load_state_dict(torch.load("models/iris_nn.pt", weights_only=True))
model.eval()

dummy = torch.randn(1, 4)
torch.onnx.export(
    model,
    dummy,
    "models/iris_nn.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)
print("iris_nn.onnx saved")

iris_nn.onnx saved
